In [1]:
import re
import json
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, box
import rasterio as rio
from rasterio.warp import transform_bounds, reproject, Resampling, calculate_default_transform
import xarray as xr
import rioxarray  # noqa: F401 (registers .rio accessor)
from pyproj import CRS, Transformer
from scipy.spatial import cKDTree
from tqdm import tqdm
import pytz
from collections import defaultdict
import math
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
import netCDF4

# --------------------------- CONFIG ---------------------------------
BASE_DIR = Path().resolve().parent  # current working dir
print("BASE_DIR:", BASE_DIR)


CONFIG = {
    "wy_start": "2024-10-01T00:00:00Z",
    "wy_end":   "2025-05-31T23:59:59Z",
    # "test_start": "2025-02-01T00:00:00Z",   # narrow test window first
    # "test_end":   "2025-04-01T00:00:00Z",
    "test_start": "2024-10-01T00:00:00Z",   # Entire window
    "test_end":   "2025-05-31T23:59:59Z",

    "station_meta_csv": BASE_DIR / "Data/Stations/station_metadata_20241001_20250531.csv",
    "station_dir": BASE_DIR / "Data/Stations",   # per-station CSVs
    "imerg_dir":   BASE_DIR / "Data/IMERG",      # parquet (wide)
    "mros_parquet": BASE_DIR / "Data/observations/wy25_mros_obs.parquet",

    "dem_path": "C:/Users/EmmaGolub/Desktop/MRoS_local/local_data/DEM_AOI_TNM_10m.tif",
    "out_dir":  BASE_DIR / "outputs/hourly_pipeline",

    "idw_power": 2.0,
    "k_nearest": 8,
    "min_points": 3,
    "lapse_K_per_m": -0.005,   # constant lapse for temps
    "proj_fallback": "EPSG:3310"  # if DEM is geographic
}

out_dir = Path(CONFIG["out_dir"])
out_dir.mkdir(parents=True, exist_ok=True)
print("out_dir:", out_dir)


BASE_DIR: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype
out_dir: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline


In [8]:
# ----------------- LOAD -----------------
st_hr   = pd.read_parquet(out_dir / "stations_hourly.parquet")
imerg_hr = pd.read_parquet(out_dir / "imerg_hourly.parquet")
mros     = pd.read_parquet(out_dir / "mros_hourly.parquet")

In [9]:
# ----------------- CLEAN HOURLY DATA -----------------
def clean_station_data(st_hr: pd.DataFrame) -> pd.DataFrame:
    st_clean = st_hr.copy()
    report = {}

    # numeric cutoffs
    cutoffs = {
        "temp_air": (-60, 55),
        "temp_dew": (-60, 35),
        "temp_wet": (-60, 35), 
    }
    for col, (lo, hi) in cutoffs.items():
        if col in st_clean.columns:
            n_total = st_clean[col].notna().sum()
            mask = (st_clean[col] < lo) | (st_clean[col] > hi)
            n_drop = int(mask.sum())
            pct = (n_drop / n_total * 100) if n_total > 0 else 0
            report[col] = (n_drop, pct)
            st_clean.loc[mask, col] = np.nan

    # consistency checks
    if {"temp_air", "temp_dew"} <= set(st_clean.columns):
        mask = st_clean["temp_dew"] > st_clean["temp_air"]
        n_total = st_clean["temp_dew"].notna().sum()
        n_drop = int(mask.sum())
        pct = (n_drop / n_total * 100) if n_total > 0 else 0
        report["dew>air"] = (n_drop, pct) # dew point must not exceed air temperature
        st_clean.loc[mask, "temp_dew"] = np.nan

    if {"temp_air", "temp_wet"} <= set(st_clean.columns):
        mask = st_clean["temp_wet"] > st_clean["temp_air"]
        n_total = st_clean["temp_wet"].notna().sum()
        n_drop = int(mask.sum())
        pct = (n_drop / n_total * 100) if n_total > 0 else 0
        report["wet>air"] = (n_drop, pct)
        st_clean.loc[mask, "temp_wet"] = np.nan

    # report
    print("Data cleaning summary:")
    for k, (n, pct) in report.items():
        print(f"  {k:8s}: dropped {n:6d} ({pct:5.2f}%)")

    return st_clean


# clean station data
st_hr_clean = clean_station_data(st_hr)


Data cleaning summary:
  temp_air: dropped    988 ( 0.18%)
  temp_dew: dropped    130 ( 0.08%)
  temp_wet: dropped    253 ( 0.08%)
  dew>air : dropped    139 ( 0.08%)
  wet>air : dropped    119 ( 0.04%)
